# Tau and Weight-Concentration Analysis — Read-Only Presentation

> **Read-only notebook**: this notebook only loads and displays the four artifacts under `results/exp_r213/`; it performs no experiment computation or file writes itself.
> Artifacts are generated by `028_exp_r213_tau_concentration.py` (data source = all 48 combinations of the frozen exp0 artifacts; purely deterministic post-processing, no randomness, CPU only).

## What this experiment addresses

At convergence, the allocation temperature $\tau \approx 0.01$ drives the per-source softmax weights close to one-hot (highly concentrated). Is this concentration a moderating variable behind the "post-correction antagonism" phenomenon (postNP performing worse than the GNN baseline)?

**Constructing inference-time evidence**: applying a power rescaling to the already-normalized weights, $w_\kappa = w^\kappa / \sum w^\kappa$ (per source), is exactly equivalent to changing the inference-time temperature to $\tau_{\mathrm{eff}} = \tau_0/\kappa$ (the power–temperature duality of softmax). $\kappa \in \{0.25, 0.5, 1, 2, 4\}$ corresponds to $\tau_{\mathrm{eff}} \in \{0.04, 0.02, 0.01, 0.005, 0.0025\}$ — $\kappa<1$ flattens (raises the temperature), $\kappa>1$ sharpens (lowers the temperature), $\kappa=1$ is the identity (the anchor point).

**Underflow diagnostic (performed first)**: under the float32 pipeline, tail weights may already be exactly zero, and rescaling with $\kappa<1$ cannot raise a weight that is already zero → we first tabulate, across all 48 combinations, the fraction of zero weights and the upper bound on truncated mass at the $\kappa=0.25$ level; only if this upper bound stays below 1% of regional mass everywhere is the power-rescaling branch permitted (otherwise we must fall back to re-running the model's forward pass under the 018 pipeline mode). The branch decision is recorded in the `branch` field of `underflow_diagnostic.json`.

**Recomputation at each κ level**: per-region RMSE (test-fold basis) for the GNN baseline plus the four post-correction arms (postN / postP / postNP / addNP, sharing the frozen numerical path from script 017); concentration metrics (normalized entropy, Gini, top-1/5/10% mass) for every (combination, region, κ, arm); and a regression of the per-region antagonism magnitude (postNP − baseline) on baseline concentration.

In [ ]:
# Read-only: load artifacts
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# CJK-capable font fallback (Windows)
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

EXP_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = EXP_DIR / 'results' / 'exp_r213'

with open(OUT / 'underflow_diagnostic.json', encoding='utf-8') as f:
    DIAG = json.load(f)
with open(OUT / 'regression.json', encoding='utf-8') as f:
    REG = json.load(f)
SWEEP = pd.read_csv(OUT / 'tau_sweep.csv')
CONC = pd.read_csv(OUT / 'concentration_metrics.csv')

META = REG['meta']
KAPPAS = META['kappas']
CONFIGS = META['configs']
ARMS = META['arms']
print('Artifact directory:', OUT)
print('branch  :', META['branch'])
print('kappa levels  :', KAPPAS, ' -> tau_eff:', [META['tau_effective'][str(k)] for k in KAPPAS])
print('tau_sweep rows:', len(SWEEP), ' concentration rows:', len(CONC))
print('Generated at:', META['timestamp'], ' numpy', META['numpy'])

## 1. Underflow Diagnostic and Branch Selection

If tail weights are already exactly zero under the float32 pipeline, power rescaling in the $\kappa<1$ direction will be systematically biased toward "staying concentrated." Diagnostic convention: a weight stored as 0 has a true value $< B = 1.4\times10^{-45}$ (the smallest positive float32 subnormal, a conservative upper bound); the upper bound on truncated mass at the $\kappa=0.25$ level is $U_r = n_{\mathrm{zero}} B^\kappa / (\sum_{w>0} w^\kappa + n_{\mathrm{zero}} B^\kappa)$.

In [ ]:
ov = DIAG['overall']
diag_tbl = pd.DataFrame([{
    'Total agents scanned': ov['total_agents_scanned'],
    'Total exact-zero weights': ov['total_zero_weights'],
    'Zero-weight fraction': f"{ov['zero_weight_frac']:.3e}",
    'Max per-source zero-weight fraction': f"{ov['max_source_zero_frac']:.3e}",
    'Max per-source truncated-mass upper bound (κ=0.25)': f"{ov['max_source_truncated_mass_ub']:.3e}",
    'Max per-region truncated-mass upper bound (κ=0.25)': f"{ov['max_location_truncated_mass_ub']:.3e}",
    'Threshold': ov['threshold'],
    'All below threshold': ov['all_below_threshold'],
}]).T.rename(columns={0: 'Value'})
display(diag_tbl)

per_combo = pd.DataFrame(DIAG['per_combo']).T
print('Zero-weight count distribution across 48 combinations: min={}, max={}, combinations with nonzero count={}'.format(
    int(per_combo['n_zero_weights'].min()), int(per_combo['n_zero_weights'].max()),
    int((per_combo['n_zero_weights'] > 0).sum())))
print('\n→ branch =', DIAG['branch'])
print('   rule:', DIAG['branch_rule'])

## 2. Anchor Report

**Anchoring pitfall (empirically observed)**: the `grid_demands` from exp0 and the frozen fold rmse.csv show GPU rerun drift (max 8.7e-5, median 2.5e-5), so $\kappa=1$ **is not strongly anchored against the frozen csv**; instead:

1. **κ=1 same-source strong anchor** (relative < 1e-9, fatal): the $\kappa=1$ level is checked against RMSE recomputed directly from the same batch of `grid_demands` — being same-source, these must match;
2. **Table 3 weak anchor** (±0.005, fatal): the $\kappa=1$ level, aggregated per the paper's convention, is checked against the Table 3 (tab:main) figures;
3. **Frozen-csv cross-check** (informational): records the magnitude of GPU rerun drift.

In [ ]:
A = REG['anchors']
ka = A['kappa1_self_anchor']
print(f"κ=1 same-source strong anchor: {ka['n_cells']} cells, max rel dev = {ka['max_rel_dev_rmse']:.3e} "
      f"(< {ka['tol']:g}) → {ka['status']}")
fz = A['frozen_csv_crosscheck']
print(f"Frozen-csv cross-check (informational): {fz['n_cells']} cells, max = {fz['max_abs_dev']:.3e}, "
      f"median = {fz['median_abs_dev']:.3e} → {fz['status']}")
t3 = A['table3_weak_anchor']
rows = [{'Cell': k, 'Table 3': v['table3_value'], 'κ=1 recomputed': round(v['recomputed_kappa1'], 4),
         'Deviation': round(v['abs_dev'], 5), 'status': v['status']} for k, v in t3['cells'].items()]
print(f"\nTable 3 weak anchor (±{t3['tol']:g}) → {t3['status']}")
display(pd.DataFrame(rows))

## 3. RMSE–κ Curve Family

One panel per config: x = $\kappa$ (log scale, with $\tau_{\mathrm{eff}}$ labeled on top), y = RMSE averaged across the 16 regions (mean over seeds first, per (config, region), following the evaluation convention). The $\kappa=1$ vertical line marks the condition used in the paper.

In [ ]:
ARM_STYLE = {
    'baseline': dict(color='#333333', marker='o', label='GNN baseline'),
    'postN':    dict(color='#1f77b4', marker='s', label='postN (multiplicative NTL)'),
    'postP':    dict(color='#2ca02c', marker='^', label='postP (multiplicative Proximity)'),
    'postNP':   dict(color='#d62728', marker='D', label='postNP (multiplicative NTL×Proximity)'),
    'addNP':    dict(color='#9467bd', marker='v', label='addNP (additive NTL×Proximity)'),
}
CONFIG_TITLE = {'baseline': 'baseline (no training prior)', 'ntl': 'ntl (NTL prior)',
                'proximity': 'proximity (Proximity prior)', 'ntl_prox': 'ntl_prox (NTL+Proximity prior)'}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, config in zip(axes.ravel(), CONFIGS):
    for arm in ARMS:
        curve = REG['kappa_curves_rmse'][config][arm]
        ys = [curve[str(k)]['rmse_mean'] for k in KAPPAS]
        st = ARM_STYLE[arm]
        ax.plot(KAPPAS, ys, color=st['color'], marker=st['marker'], label=st['label'])
    ax.axvline(1.0, color='grey', ls='--', lw=0.8)
    ax.set_xscale('log')
    ax.set_xticks(KAPPAS)
    ax.set_xticklabels([f'{k:g}' for k in KAPPAS])
    tau_lbls = [f"{META['tau_effective'][str(k)]:g}" for k in KAPPAS]
    sec = ax.secondary_xaxis('top')
    sec.set_xticks(KAPPAS)
    sec.set_xticklabels(tau_lbls)
    sec.set_xlabel(r'$\tau_{\rm eff}$', fontsize=9)
    ax.set_title(CONFIG_TITLE[config])
    ax.set_xlabel(r'$\kappa$ (power rescaling exponent)')
    ax.set_ylabel('RMSE (mean over 16 regions)')
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.suptitle('RMSE–κ curve family: κ=1 is the paper condition (τ=0.01), κ<1 flattens / κ>1 sharpens', y=1.02)
fig.tight_layout()
plt.show()

## 4. Antagonism Magnitude vs. κ

Antagonism magnitude $\Delta = \mathrm{RMSE}(\mathrm{postNP}) - \mathrm{RMSE}(\mathrm{baseline})$ (per-region, averaged across the 16 regions after taking the seed mean). $\Delta > 0$ means "the correction makes the GNN worse" (the antagonism phenomenon discussed in the paper).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
CONFIG_COLOR = {'baseline': '#d62728', 'ntl': '#1f77b4',
                'proximity': '#2ca02c', 'ntl_prox': '#9467bd'}
for config in CONFIGS:
    entry = REG['headline']['antagonism_by_kappa'][config]
    means = [entry[str(k)]['delta_np_mean'] for k in KAPPAS]
    stds = [entry[str(k)]['delta_np_std_ddof0'] for k in KAPPAS]
    ax.errorbar(KAPPAS, means, yerr=stds, color=CONFIG_COLOR[config],
                marker='o', capsize=3, label=CONFIG_TITLE[config])
ax.axhline(0, color='grey', lw=0.8)
ax.axvline(1.0, color='grey', ls='--', lw=0.8)
ax.set_xscale('log')
ax.set_xticks(KAPPAS)
ax.set_xticklabels([f'{k:g}' for k in KAPPAS])
ax.set_xlabel(r'$\kappa$ (← flatten · sharpen →)')
ax.set_ylabel(r'$\Delta$ RMSE (postNP − baseline)')
ax.set_title('Antagonism magnitude vs. κ (>0 = correction makes the GNN worse)')
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

for config in CONFIGS:
    t = REG['headline']['antagonism_vs_kappa_trend'][config]
    print(f"{config:10s}: Spearman(κ, mean Δ) = {t['spearman_rho_vs_kappa']:+.3f} "
          f"→ {t['direction']}  (monotone nondecreasing: {t['monotone_nondecreasing']})")

## 5. Antagonism vs. Concentration Scatter

Each point = one (config, region) pair (seed mean). x = **baseline concentration** (Gini of the baseline arm's demand distribution), y = antagonism magnitude $\Delta$. Dashed line = pooled OLS fit (`regression.json`). Split into five panels by κ.

In [ ]:
# Reconstruct scatter data from the CSVs (read-only; same convention as build_regressions in 028)
loc_avg = (SWEEP.groupby(['config', 'arm', 'kappa', 'location'])['rmse']
           .mean().reset_index())
wide = loc_avg.pivot_table(index=['config', 'kappa', 'location'],
                           columns='arm', values='rmse').reset_index()
wide['delta_np'] = wide['postNP'] - wide['baseline']
base_conc = (CONC[CONC['arm'] == 'baseline']
             .groupby(['config', 'kappa', 'location'])[['gini']].mean().reset_index())
pts = wide.merge(base_conc, on=['config', 'kappa', 'location'])

fig, axes = plt.subplots(1, len(KAPPAS), figsize=(16, 3.4), sharey=True)
for ax, kappa in zip(axes, KAPPAS):
    sub = pts[pts['kappa'] == kappa]
    for config in CONFIGS:
        s = sub[sub['config'] == config]
        ax.scatter(s['gini'], s['delta_np'], s=18, alpha=0.75,
                   color=CONFIG_COLOR[config], label=CONFIG_TITLE[config])
    r = REG['regressions'][str(kappa)]['pooled_all_configs']['gini']
    xs = np.linspace(sub['gini'].min(), sub['gini'].max(), 50)
    ax.plot(xs, r['intercept'] + r['slope'] * xs, 'k--', lw=1)
    ax.axhline(0, color='grey', lw=0.6)
    ax.set_title(f"κ={kappa:g}\nslope={r['slope']:+.2f}, r={r['pearson_r']:+.2f}, "
                 f"p={r['p_value']:.2g}", fontsize=9)
    ax.set_xlabel('Baseline Gini')
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r'$\Delta$ RMSE (postNP − baseline)')
axes[0].legend(fontsize=7)
fig.suptitle('Antagonism vs. concentration scatter (pooled fit line; each point = config × region, seed mean)', y=1.08)
fig.tight_layout()
plt.show()

## 6. Concentration Metrics vs. κ (Baseline Arm)

The baseline arm's normalized entropy is **non-strictly monotonically decreasing** in κ (a classical result for escort distributions: $dH/d\kappa = -\kappa\,\mathrm{Var}_{p_\kappa}(\log w) \le 0$) — this is also one of the hard assertions in `test_r213`. The correction arms are not guaranteed to be monotonic (their logits carry a $\log f$ offset), so no assertion is made for them; they are shown for reference only.

In [ ]:
base = CONC[CONC['arm'] == 'baseline']
summ = base.groupby('kappa')[['entropy_norm', 'gini', 'top1_share',
                              'top5_share', 'top10_share']].mean().round(4)
summ.index = [f'κ={k:g} (τ_eff={META["tau_effective"][str(k)]:g})' for k in summ.index]
print('Baseline-arm concentration metrics (mean over 192 combination-region pairs):')
display(summ)

viol = 0
for key, grp in base.groupby(['seed', 'config', 'fold', 'location']):
    ent = grp.sort_values('kappa')['entropy_norm'].to_numpy()
    if not (np.diff(ent) <= 1e-12).all():
        viol += 1
print(f'Entropy monotonicity violations (baseline arm, tolerance 1e-12): {viol} / 192 combination-region pairs')

## 7. Conclusions (generated programmatically from the numeric results)

In [ ]:
# All conclusion lines are generated programmatically from the saved numbers; none are hand-written
lines = []
lines.append(f"[Branch] Underflow diagnostic: total zero weights = {ov['total_zero_weights']}, "
             f"max truncated-mass upper bound = {ov['max_source_truncated_mass_ub']:.3e} "
             f"(< {ov['threshold']}) → branch = {DIAG['branch']}")
ka = REG['anchors']['kappa1_self_anchor']
t3 = REG['anchors']['table3_weak_anchor']
lines.append(f"[Anchor] κ=1 same-source strong anchor {ka['status']} (max {ka['max_rel_dev_rmse']:.1e} < {ka['tol']:g}); "
             f"Table 3 weak anchor {t3['status']} (7/7 cells ±{t3['tol']:g})")
for config in CONFIGS:
    t = REG['headline']['antagonism_vs_kappa_trend'][config]
    d_at = t['delta_np_means_by_kappa']
    lines.append(f"[Antagonism-κ] {config}: Δ(κ=0.25)={float(d_at['0.25']):+.3f} → "
                 f"Δ(κ=1)={float(d_at['1.0']):+.3f} → Δ(κ=4)={float(d_at['4.0']):+.3f}, "
                 f"Spearman={t['spearman_rho_vs_kappa']:+.2f} → {t['direction']}")
for p, s in REG['headline']['concentration_slope_at_kappa1'].items():
    lines.append(f"[Antagonism-concentration κ=1 pooled] predictor={p}: slope={s['slope']:+.3f}, "
                 f"r={s['pearson_r']:+.3f}, p={s['p_value']:.3g} → {s['direction']}")
for ln in lines:
    print(ln)